# 🎯 [MNT06_M08] 복잡 형상 부품 50종 일괄 신뢰성 평가 시뮬레이션 부하 테스트
### 실습 솔루션 & 정답 해설 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/MNT06/mnt06_m08_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 장비정비·수리부속 (Level 3) |
| **미션 ID** | MNT06_M08 |
| **미션 주제** | 복잡 형상 부품 50종 일괄 신뢰성 평가 시뮬레이션 부하 테스트 |
| **소요 시간** | 약 3시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
수리부속의 폐기율이 높으면 정비비용과 재고 부담이 커지고, 조달이 늦어지는 동안 장비는 가동하지 못한다. 폐기가 잦은 부품 가운데 야전에서 3D 프린팅으로 대체 제작할 수 있는 부품을 가려내면 폐기 손실과 조달 대기를 함께 줄일 수 있다. 본 프로젝트는 부품 폐기율의 산정·예측을 대체 제작 타당성 판단과 연결하는 것을 목표로, 교육용 가상 부품 자료(CAD 요약 치수, 금속·폴리머 소재 물성, 사용 하중, 폐기율, 조달 소요일)를 코드로 만들어 분석한다. 데이터 정의와 메쉬·슬라이싱 전처리, 형상 기반 Baseline과 탐색적 분석, 인장강도·피로 수명 예측과 군 규격 부합 신뢰도 점수, 대시보드와 설명가능성, 정비기술관 승인(HITL) 시뮬레이터, 50종 일괄 평가 스트레스 시험, 파괴 시험 대비 AAR 재보정과 효과 산출, 재학습을 포함한 통합 도구까지 단계별로 구현한다. 최종 목표는 폐기 손실이 큰 부품 중 신뢰성이 확인된 대체 제작 후보를 근거와 함께 제시하고, 실제 장착 여부는 정비기술관이 결정하도록 지원하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📁 2. 실습 산출물(Deliverables) 디렉터리 구조 및 안내

> **💡 [학습 안내] 산출물 디렉터리 아키텍처**  
> 본 과제는 단순 일회성 노트북 실행에 그치지 않고, **실제 군수 정비 야전에 즉시 도입 가능한 "완성형 국방 3D 프린팅 대체제작 타당성 및 부품 폐기율 예측 도구"를 완성하는 실무 엔지니어링 과정**입니다. 각 미션의 **산출물(Deliverables)**에 명시된 디렉터리 경로는 실무 표준 소프트웨어 아키텍처에 해당합니다:

| 디렉토리 | 군 실무 소프트웨어 아키텍처 분류 | 주요 보관 산출물 | 관련 미션 |
| :--- | :--- | :--- | :---: |
| **`docs/`** | **국방 도메인 및 데이터 규격서** | 군수 도메인 분석서 및 데이터 딕셔너리(`docs/mnt06_logistics_domain_spec.md`), 표준 운영 절차 기술 매뉴얼(`docs/mnt06_operations_guide.md`) | 미션 1, 10 |
| **`src/etl/`** | **CAD 메쉬 정제 및 전처리 모듈** | 3D CAD 메쉬 정제, 슬라이싱 데이터 파싱 파이프라인(`src/etl/mnt06_data_cleansing.py`) | 미션 2 |
| **`notebooks/`** | **탐색적 데이터 분석(EDA) & Baseline** | 기하 형상 기반 1차 제작 가능 여부 판별 Baseline(`notebooks/mnt06_eda_analysis.ipynb`) | 미션 3 |
| **`src/features/`** | **FEA 연계 피처 엔지니어링** | 유한요소해석 연계 응력 계산 및 적층 공정 피처 엔지니어링(`src/features/mnt06_feature_engineering.py`) | 미션 4 |
| **`src/models/`** | **AI 예측 & 군 규격 신뢰도 엔진** | 인장강도·피로내구도 예측 및 MIL-STD 규격 부합 신뢰도 엔진(`src/models/mnt06_advanced_model.py`) | 미션 5 |
| **`src/explainability/`** | **XAI 설명가능성 모듈** | 소요 시간·비용 분석 및 신뢰도 평가 결과 XAI 원인 설명기(`src/explainability/mnt06_xai_explainer.py`) | 미션 6 |
| **`src/optimization/`** | **HITL 승인 절차 시뮬레이터** | 정비기술관 HITL 검토·승인 큐 및 야전 부품 실장착 시뮬레이터(`src/optimization/mnt06_simulator.py`) | 미션 7 |
| **`src/dashboard/`** | **스트레스 시험 & 관제 대시보드** | 복잡 형상 부품 50종 일괄 신뢰성 평가 시뮬레이션 대시보드(`src/dashboard/mnt06_app.py`) | 미션 8 |
| **`reports/`** | **검증 및 AAR 리포트** | 사후검토(AAR) 파괴 시험 결과 오차 분석 및 ROI 보고서(`reports/mnt06_roi_analysis.md`) | 미션 9 |
| **`pipeline/`** | **MLOps 자동 재학습** | 실물 피드백 기반 자동 재학습 및 E2E 통합 평가 파이프라인(`pipeline/mnt06_retrain.py`) | 미션 10 |

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 08. 복잡 형상 부품 50종 일괄 신뢰성 평가 시뮬레이션 부하 테스트
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

복잡 형상 부품 50종 일괄 신뢰성 평가 시뮬레이션 부하 테스트를 수행한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **복잡 형상 부품 50종(최대 500종)을 일괄 평가하는 상황에서 신뢰성 평가와 대시보드가 제때 동작하는지 부하 테스트한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **복잡 형상 부품 50·100·200·500종 일괄 요청에 손상 레코드와 모델 장애를 주입하고 처리와 화면 생성 시간, 유실 여부, 필터 조회 응답을 측정 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/dashboard/mnt06_app.py (Streamlit/Dash 등)` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 의사결정 지원에 최적화된 시각적 위계 및 사용자 경험(UX)

#### 미션 PASS 조건

- 정량 기준: 대시보드 로딩 시간 2초 이내, 경보 발령 정확도 95% 이상


- 진행 기준: 웹 기반 분석 대시보드 구동 및 실시간 필터링 기능 검증


- 검증 방법: 실제 군수 데이터 연동 화면 테스트 및 UI 동작 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
# ============================================================================
# [미션 08 산출물 파일 연계]: src/dashboard/mnt06_app.py
# (선택) 로컬 모듈 파일로 추출 저장 시: %%writefile src/dashboard/mnt06_app.py
# ============================================================================
"""복잡 형상 부품 50종(최대 500종) 일괄 신뢰성 평가를 요청하면서 손상 레코드 주입과 평가 모델 장애를 일으켜, 일괄 처리+화면 생성 시간, '신뢰성 미달 경보' 정확도, 요청 유실, 손상 데이터의 수동 확인 분리, 필터 조회 응답, 모델 장애 시 간이 규칙으로의 기능 축소 운용을 스트레스 테스트한다."""
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor

# ---------------------------------------------------------------------------
# [교육용가상자료] 야전 3D 프린팅 대체 제작 후보 부품 생성기 (실제 부품·재고·규격 값 아님)
# 폐기율이 높은 수리부속 중 3D 프린팅으로 대체 제작할 수 있는지 판단하기 위한 가상 부품 자료를 만든다.
# 소재 물성·치수·하중은 교육용으로 단순화한 값이며 실제 시험 성적서나 군 규격 값과 무관하다.
# ---------------------------------------------------------------------------
MAT = {  # 소재: (공정, 기준 인장강도 MPa, 밀도 g/cm3, 소재 단가 만원/kg, 출력비 만원/h, 최소 벽두께 mm, 출력 속도 cm3/h)
    "PA12": ("polymer", 45, 1.0, 8, 3, 0.8, 18), "PETG": ("polymer", 50, 1.27, 5, 2, 0.8, 15),
    "CF-PA": ("polymer", 70, 1.2, 14, 3, 1.0, 12), "AlSi10Mg": ("metal", 350, 2.67, 30, 25, 0.5, 4),
    "SS316L": ("metal", 520, 7.9, 28, 25, 0.6, 3.5), "Ti64": ("metal", 900, 4.4, 90, 30, 0.6, 3)}
CATS = ["bracket", "housing", "clip", "gear", "duct", "handle"]
CAT_SCRAP = {"bracket": 0.04, "housing": 0.05, "clip": 0.12, "gear": 0.09, "duct": 0.08, "handle": 0.10}   # 부품군별 기본 월 폐기율
BUILD_MM = {"polymer": 300, "metal": 200}                                                                 # 가상 출력 가능 최대 치수


def make_parts(n=300, seed=0):
    """가상 부품 n종: 형상(부피·벽두께·돌출 비율), 소재, 사용 하중, 폐기율과 조달 정보를 만든다."""
    rng = np.random.default_rng(seed)
    mat = rng.choice(list(MAT), n, p=[0.2, 0.15, 0.1, 0.2, 0.2, 0.15])
    proc = np.array([MAT[m][0] for m in mat])
    vol = np.clip(rng.lognormal(np.log(30), 0.9, n), 2, 800)
    aspect = rng.lognormal(0.3, 0.4, n)
    max_dim = 10 * vol ** (1 / 3) * aspect ** (1 / 3) * rng.uniform(1.0, 1.6, n)
    wall = np.clip(rng.lognormal(np.log(2.0), 0.6, n), 0.3, 8)
    ovh = rng.beta(2, 5, n)
    is_poly = proc == "polymer"
    df = pd.DataFrame({
        "part_id": [f"P{i:03d}" for i in range(n)], "cat": rng.choice(CATS, n, p=[0.2, 0.15, 0.2, 0.1, 0.15, 0.2]), "material": mat, "process": proc,
        "volume_cm3": vol, "max_dim_mm": max_dim, "area_cm2": 6 * vol ** (2 / 3) * rng.uniform(1.0, 2.2, n), "min_wall_mm": wall, "overhang": ovh,
        "n_holes": rng.poisson(2.0, n), "thread": rng.random(n) < 0.25, "orient_deg": rng.uniform(0, 90, n),
        "layer_mm": np.where(is_poly, rng.choice([0.1, 0.2, 0.3], n), rng.choice([0.03, 0.05], n)),
        "infill": np.where(is_poly, rng.choice([40, 60, 80, 100], n, p=[0.15, 0.3, 0.3, 0.25]), 100),
        "load_N": np.clip(rng.lognormal(np.where(is_poly, np.log(230), np.log(2300)), 0.7, n), 10, 20000), "temp_C": rng.uniform(-20, 90, n),
        "crit": rng.choice([1, 2, 3], n, p=[0.35, 0.4, 0.25]), "cert": rng.choice([0, 1], n, p=[0.45, 0.55]),   # 소재 인증서 보유
        "import_part": rng.random(n) < 0.4})
    df["scrap_rate"] = np.clip(df["cat"].map(CAT_SCRAP) * rng.lognormal(0, 0.35, n) * (1 + 0.5 * (df["load_N"] > 800)) * (1 + 0.3 * (df["temp_C"] > 60)), 0.005, 0.5)
    df["lead_days"] = np.clip(rng.lognormal(np.log(25), 0.5, n) * np.where(df["import_part"], 2.2, 1.0), 5, 300)
    # ---- 숨은 진실값(실제 시험 결과에 해당): 모델은 위 입력으로 이것을 예측한다 ----
    eff = np.where(is_poly, 0.55 + 0.45 * df["infill"] / 100, 1 - 0.3 * ovh * 0.2)
    aniso = np.where(is_poly, 1 - 0.3 * np.sin(np.radians(df["orient_deg"])) ** 2, 1 - 0.08 * np.sin(np.radians(df["orient_deg"])))
    heat = np.where(is_poly, 1 - 0.35 * np.clip((df["temp_C"] - 40) / 80, 0, 1), 0.98)
    base = np.array([MAT[m][1] for m in mat])
    df["strength_mpa"] = base * eff * aniso * heat * rng.lognormal(0, 0.06, n)
    cross = df["min_wall_mm"] * np.sqrt(df["max_dim_mm"]) * 1.0                                  # 하중 지지 단면적 mm2(단순화)
    df["stress_mpa"] = df["load_N"] / cross
    df["safety"] = df["strength_mpa"] / df["stress_mpa"]
    ratio = np.clip(df["stress_mpa"] / df["strength_mpa"], 0, 1.5)
    df["log_cycles"] = np.clip(7.5 - 6.0 * ratio + rng.normal(0, 0.25, n), 2, 7.5)               # 피로 수명 log10(회)
    support = 1 + 0.5 * ovh
    df["build_h"] = vol * support / np.array([MAT[m][6] for m in mat]) + 0.5
    mass_kg = vol * np.array([MAT[m][2] for m in mat]) * support / 1000
    df["print_cost"] = mass_kg * np.array([MAT[m][3] for m in mat]) + df["build_h"] * np.array([MAT[m][4] for m in mat]) + 3
    df["buy_cost"] = df["print_cost"] * rng.lognormal(0.25, 0.45, n) + 5                      # 구매 단가(만원, 가상)
    wall_ok = df["min_wall_mm"] >= np.array([MAT[m][5] for m in mat]) * rng.uniform(0.95, 1.25, n)
    dim_ok = df["max_dim_mm"] <= np.where(is_poly, BUILD_MM["polymer"], BUILD_MM["metal"])
    warp = (~is_poly) & (df["max_dim_mm"] > 140) & (df["overhang"] > 0.3)                     # 금속 대형 오버행 변형
    thr_bad = df["thread"] & (df["min_wall_mm"] < 1.5) & is_poly                              # 얇은 나사부 파손
    df["printable"] = wall_ok & dim_ok & (ovh < 0.5) & ~warp & ~thr_bad
    risk = np.clip(1 - df["safety"] / 3, 0, 1)
    df["conform"] = np.clip(100 * (0.45 * np.minimum(df["safety"] / 3, 1) + 0.25 * np.clip((df["log_cycles"] - 3) / 4.5, 0, 1)
                                   + 0.15 * (1 - np.clip(ovh * 1.2, 0, 1)) + 0.15 * df["cert"]) - 8 * risk * df["crit"] / 3 + rng.normal(0, 2, n), 0, 100)
    df["demand_month"] = rng.poisson(np.where(df["cat"].isin(["clip", "handle"]), 40, 15)) + 1               # 월 소요수량(가상)
    df["scrap_cost"] = df["scrap_rate"] * df["demand_month"] * df["buy_cost"]                              # 월 폐기 손실액(만원, 가상)
    return df

# ---- 단순화한 해석(FEA 대용) 값과 파생 피처: 입력 정보만으로 계산한다(시험 결과 누설 없음) ----
def make_features(d):
    """간이 응력 해석(응력집중계수 포함)과 소재·공정 지식에서 만든 파생 피처 16개."""
    X = pd.DataFrame(index=d.index)
    base = d["material"].map(lambda m: MAT[m][1]).astype(float)
    X["base_strength"] = base
    X["is_metal"] = (d["process"] == "metal").astype(int)
    X["cross_mm2"] = d["min_wall_mm"] * np.sqrt(d["max_dim_mm"])
    X["kt"] = 1 + 0.15 * d["n_holes"] + 0.5 * d["thread"].astype(int)                   # 구멍·나사부 응력집중계수(간이)
    X["fea_stress"] = d["load_N"] / X["cross_mm2"] * X["kt"]                             # 간이 해석 응력(MPa)
    X["stress_ratio"] = X["fea_stress"] / base                                          # 기준강도 대비 응력비
    X["infill_eff"] = np.where(X["is_metal"] == 1, 1.0, 0.55 + 0.45 * d["infill"] / 100)
    X["orient_sin2"] = np.sin(np.radians(d["orient_deg"])) ** 2
    X["temp_derate"] = np.where(X["is_metal"] == 1, 0.0, np.clip((d["temp_C"] - 40) / 80, 0, 1))
    X["overhang_layer"] = d["overhang"] * d["layer_mm"]
    X["slender"] = d["max_dim_mm"] / d["min_wall_mm"]
    X["surf_vol"] = d["area_cm2"] / d["volume_cm3"]
    X["wall_ratio"] = d["min_wall_mm"] / d["material"].map(lambda m: MAT[m][5])
    X["load_per_vol"] = d["load_N"] / d["volume_cm3"]
    X["cert"] = d["cert"].astype(int)
    X["crit"] = d["crit"]
    return X

ALERT, BAND = 70.0, 5.0     # 점수 65 미만 '경보', 75 이상 '정상', 그 사이는 모델 불확실 구간이라 '보류(수동 확인)'


def make_batch(n, seed):
    """[교육용가상자료] 복잡 형상(돌출 많고 구멍 3개 이상) 부품을 n종 뽑아 한꺼번에 평가를 요청한다."""
    pool = make_parts(max(4 * n, 600), seed=seed)
    cx = pool[(pool["overhang"] > 0.3) & (pool["n_holes"] >= 3)]
    return cx.sample(n, replace=len(cx) < n, random_state=seed).reset_index(drop=True)


def validate(r):
    errs = []
    if not r["min_wall_mm"] > 0 or not r["max_dim_mm"] > 0:
        errs.append("치수 오류")
    if r["material"] not in MAT:
        errs.append("소재 코드 오류")
    if not np.isfinite(r["load_N"]):
        errs.append("하중 누락")
    return errs


def fallback_score(X):
    """모델 장애 시 간이 규칙: 응력/기준강도 비로만 점수화(보수적)."""
    return np.clip(100 * (1 - 1.8 * X["stress_ratio"]), 0, 100).values


def evaluate(batch, model, fail_model=False):
    ok_mask = np.array([not validate(r) for r in batch.to_dict("records")])
    good, manual = batch[ok_mask].copy(), batch[~ok_mask]
    X = make_features(good)
    try:
        if fail_model:
            raise RuntimeError("model down")
        good["score"], good["src"] = np.clip(model.predict(X), 0, 100), "모델"
    except Exception:
        good["score"], good["src"] = fallback_score(X), "간이 규칙"
    good["state"] = np.where(good["score"] < ALERT - BAND, "경보", np.where(good["score"] >= ALERT + BAND, "정상", "보류"))
    good["alert"] = good["state"] == "경보"
    return good, manual


def dashboard(good):
    fig, ax = plt.subplots(1, 3, figsize=(14, 4))
    ax[0].hist(good["score"], bins=20); ax[0].axvline(ALERT, ls="--", c="r"); ax[0].set_title("Score distribution")
    ax[1].scatter(good["score"], good["build_h"], c=good["state"].map({"경보": "red", "보류": "orange", "정상": "gray"}), s=12); ax[1].set_yscale("log"); ax[1].set_title("Score vs build hours (red = alert, orange = hold)")
    good.groupby("cat")["alert"].sum().plot.bar(ax=ax[2]); ax[2].set_title("Alerts by category")
    fig.tight_layout(); fig.canvas.draw()
    return fig


hist = make_parts(1000, seed=11)
model = GradientBoostingRegressor(n_estimators=150, max_depth=3, learning_rate=0.06, random_state=0).fit(make_features(hist), hist["conform"])
rows, keep = [], {}
for n, bad, down in [(50, 0, False), (50, 4, False), (100, 6, False), (200, 10, False), (500, 20, False), (50, 4, True)]:
    batch = make_batch(n, seed=n + bad)
    for k in range(bad):                                                   # 손상 레코드 주입
        i = k * 3
        if k % 3 == 0:
            batch.loc[i, "min_wall_mm"] = -1.0
        elif k % 3 == 1:
            batch.loc[i, "material"] = "unknown"
        else:
            batch.loc[i, "load_N"] = np.nan
    t0 = time.perf_counter()
    good, manual = evaluate(batch, model, fail_model=down)
    fig = dashboard(good)
    sec = time.perf_counter() - t0
    truth = good["conform"] < ALERT
    decided = good["state"] != "보류"
    acc = float((good["alert"] == truth)[decided].mean())
    rows.append({"상황": f"{n}종" + (f", 손상 {bad}건" if bad else "") + (", 모델 장애" if down else ""), "평가": len(good), "수동 확인": len(manual),
                 "유실": n - len(good) - len(manual), "처리+화면(초)": round(sec, 2), "경보": int(good["alert"].sum()), "보류": int((~decided).sum()), "경보 정확도": round(acc, 3),
                 "미달 놓침": int((truth & (good["state"] == "정상")).sum())})
    keep[n, bad, down] = (fig, good)
    if (n, bad, down) != (50, 4, False):
        plt.close(fig)
T = pd.DataFrame(rows)
print("[교육용가상자료] 복잡 형상 부품 일괄 신뢰성 평가 스트레스 테스트")
print(T.to_string(index=False))
m = T.iloc[1]
normal = T[~T["상황"].str.contains("장애")]
print(f"\n50종(손상 4건) 기준: 처리+화면 {m['처리+화면(초)']}초 (목표 2초 이내: {'달성' if normal['처리+화면(초)'].max() <= 2 else '미달'}), "
      f"확정 판정 경보 정확도 {normal['경보 정확도'].min():.1%}~{normal['경보 정확도'].max():.1%} (목표 95% 이상: {'달성' if normal['경보 정확도'].min() >= 0.95 else '미달'}), 유실 {int(T['유실'].sum())}건")
print(f"보류(65~75점) 비율 {(T['보류'] / T['평가']).min():.0%}~{(T['보류'] / T['평가']).max():.0%}: 모델이 판단하기 어려운 구간은 자동 판정하지 않고 정비기술관 수동 확인으로 넘긴다")
print(f"한계치: 500종에서도 {T.iloc[4]['처리+화면(초)']}초 -> 이 규모에서는 처리 시간이 병목이 아니다. 병목은 손상 레코드 수동 확인과 정비기술관 검토 인력이다.")
print("손상 레코드는 삭제하지 않고 '수동 확인' 목록으로 넘겨 설계자가 CAD·시험 기록을 재확인한다.")

good = keep[50, 4, False][1]
qt = []
for c in good["cat"].unique():
    for mt in MAT:
        t0 = time.perf_counter(); good[(good["cat"] == c) & (good["material"] == mt)]; qt.append((time.perf_counter() - t0) * 1000)
print(f"실시간 필터(부품군 x 소재) 평균 {np.mean(qt):.2f} ms")
fb = T.iloc[5]
print(f"\n모델 장애 시 간이 규칙으로 전환: 확정 판정 정확도 {fb['경보 정확도']:.1%}, 경보 {fb['경보']}건(정상 모델 {T.iloc[1]['경보']}건), 미달 놓침 {fb['미달 놓침']}건(정상 모델 {T.iloc[1]['미달 놓침']}건)")
print("기능 축소 운용: 모델 장애 또는 처리 2초 초과 시 응력/기준강도 비 규칙 점수로 즉시 평가하고, 정밀 점수는 복구 후 재계산한다. 규칙 점수는 보수적이라 경보가 늘어난다.")
plt.show()


<details>
<summary><b>💡 [참고] 미션 08 예상 실행 결과 (Output)</b></summary>

```text
[교육용가상자료] 복잡 형상 부품 일괄 신뢰성 평가 스트레스 테스트
               상황  평가  수동 확인  유실  처리+화면(초)  경보  보류  경보 정확도  미달 놓침
              50종  50      0   0      0.88  29   9   0.976      1
       50종, 손상 4건  46      4   0      0.67  26  11   0.971      1
      100종, 손상 6건  94      6   0      0.78  67  10   1.000      0
     200종, 손상 10건 190     10   0      0.57 104  35   0.987      0
     500종, 손상 20건 480     20   0      0.50 262 110   0.978      0
50종, 손상 4건, 모델 장애  46      4   0      0.61  40   4   0.810      0

50종(손상 4건) 기준: 처리+화면 0.67초 (목표 2초 이내: 달성), 확정 판정 경보 정확도 97.1%~100.0% (목표 95% 이상: 달성), 유실 0건
보류(65~75점) 비율 9%~24%: 모델이 판단하기 어려운 구간은 자동 판정하지 않고 정비기술관 수동 확인으로 넘긴다
한계치: 500종에서도 0.5초 -> 이 규모에서는 처리 시간이 병목이 아니다. 병목은 손상 레코드 수동 확인과 정비기술관 검토 인력이다.
손상 레코드는 삭제하지 않고 '수동 확인' 목록으로 넘겨 설계자가 CAD·시험 기록을 재확인한다.
실시간 필터(부품군 x 소재) 평균 0.72 ms

모델 장애 시 간이 규칙으로 전환: 확정 판정 정확도 81.0%, 경보 40건(정상 모델 26건), 미달 놓침 0건(정상 모델 1건)
기능 축소 운용: 모델 장애 또는 처리 2초 초과 시 응력/기준강도 비 규칙 점수로 즉시 평가하고, 정밀 점수는 복구 후 재계산한다. 규칙 점수는 보수적이라 경보가 늘어난다.
```
</details>
